In [2]:
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, auc, f1_score, hamming_loss
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

In [3]:
import sys
from pathlib import Path
sys.path.insert(0, str(next(p for p in (Path.cwd(), *Path.cwd().parents)
                            if (p / "scripts" / "project.py").exists()) / "scripts"))
from project import *  # ROOT, CACHE_FILE, LEAD_SET, MODELS_DIR, RESULTS_DIR, load_cache, ...
MODELS_DIR, RESULTS_DIR = output_dirs()
X_train, y_train, X_test, y_test, class_names, X_feat_train, X_feat_test = load_cache()

Loading cached dataset from ptbxl_preprocessed_cache.npz...
Loaded successfully in seconds!
Signals   -> Train: (19230, 1000, 3) | Test: (2158, 1000, 3)
Features  -> Train: (19230, 36) | Test: (2158, 36)
Targets   -> Classes (5): ['CD', 'HYP', 'MI', 'NORM', 'STTC']


In [4]:
all_predictions = {}

# --- 1. Multi-Output XGBoost ---
print("Training XGBoost Multi-Output Baseline...")
xgb_clf = MultiOutputClassifier(
    xgb.XGBClassifier(
        n_estimators=150,
        max_depth=5,
        learning_rate=0.08,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    ),
    n_jobs=-1
)
xgb_clf.fit(X_feat_train, y_train)

# Predict probabilities for each class
xgb_probs = np.array(xgb_clf.predict_proba(X_feat_test))[:, :, 1].T
all_predictions["XGBoost (Tabular Limb)"] = xgb_probs
print(f"XGBoost Macro AUC: {roc_auc_score(y_test, xgb_probs, average='macro'):.4f}")

# --- 2. Multi-Output Random Forest ---
print("\nTraining Random Forest Multi-Output Baseline...")
rf_clf = MultiOutputClassifier(
    RandomForestClassifier(
        n_estimators=150,
        max_depth=12,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1
    ),
    n_jobs=-1
)
rf_clf.fit(X_feat_train, y_train)

rf_probs = np.array(rf_clf.predict_proba(X_feat_test))[:, :, 1].T
all_predictions["Random Forest (Tabular Limb)"] = rf_probs
print(f"Random Forest Macro AUC: {roc_auc_score(y_test, rf_probs, average='macro'):.4f}")

Training XGBoost Multi-Output Baseline...
XGBoost Macro AUC: 0.7952

Training Random Forest Multi-Output Baseline...
Random Forest Macro AUC: 0.7862


In [5]:
import numpy as np
import joblib

# Save predictions for the presentation plots
np.savez_compressed(RESULTS_DIR / "preds_classical.npz", xgb=xgb_probs, rf=rf_probs)

# Save the classical models (weights/trees)
joblib.dump(xgb_clf, MODELS_DIR / "xgb_model.joblib")
joblib.dump(rf_clf, MODELS_DIR / "rf_model.joblib")
print("Saved Classical predictions and models!")

Saved Classical predictions and models!
